# ⚡ GCP BigLake REST Catalog PySpark & Spark SQL Query Guide

이 주피터 노트북은 GCP **BigLake REST Catalog**(`https://biglake.googleapis.com/iceberg/v1/restcatalog`)에 적재된 Apache Iceberg 테이블(`default.external_weblog`)을 **PySpark 및 Spark SQL**을 사용하여 조회, 분석, Spark SQL DML(INSERT, UPDATE, DELETE) 수행 및 Iceberg Maintenance Procedure (`rewrite_data_files`, `expire_snapshots`)를 수행하는 엔터프라이즈 통합 가이드입니다.

### 🎯 주요 특징
1. **Zero Hardcoded Environment**: 하드코딩된 개인화/특정 프로젝트 문자열 없이 사용자 환경 변수(`GCP_PROJECT_ID`, `GCS_BUCKET_NAME`) 및 `google.auth.default()`를 통해 동적으로 자동 감지합니다.
2. **GCS Hadoop Connector Direct Integration**: ADC(`application_default_credentials.json`) 인증 정보를 바탕으로 Spark Hadoop GCS 커넥터를 자동 구성합니다.
3. **Optimized JVM & Spark Engine Parameters**: JDK 17/21+ 환경과의 완벽한 호환성을 위한 JVM 모듈 개방 플래그 및 Spark WholeStageCodegen 최적화 설정을 포함합니다.
4. **Spark SQL Native DML & Maintenance Support**: `INSERT INTO`, `UPDATE`, `DELETE FROM` SQL 구문을 통한 DML 조작 및 Compaction (`rewrite_data_files`), Snapshot Expire (`expire_snapshots`) 실행 지원.


In [ ]:
import os
import json
import google.auth
import google.auth.transport.requests

# 1. 시스템 JAVA_HOME 설정 (PySpark와 Java 26 간 호환성 오류 방지를 위해 JDK 21/17 지정)
possible_java_homes = [
    "/usr/lib/jvm/java-21-openjdk-amd64",
    "/usr/lib/jvm/java-17-openjdk-amd64",
    "/usr/lib/jvm/default-java"
]
for j_home in possible_java_homes:
    if os.path.exists(j_home):
        os.environ["JAVA_HOME"] = j_home
        os.environ["PATH"] = f"{j_home}/bin:" + os.environ.get("PATH", "")
        break

# 2. JDK 17/21+ 호환성을 위한 JVM Open Modules 설정
jvm_flags = (
    '--add-opens=java.base/java.lang=ALL-UNNAMED '
    '--add-opens=java.base/java.lang.invoke=ALL-UNNAMED '
    '--add-opens=java.base/java.lang.reflect=ALL-UNNAMED '
    '--add-opens=java.base/java.io=ALL-UNNAMED '
    '--add-opens=java.base/java.net=ALL-UNNAMED '
    '--add-opens=java.base/java.nio=ALL-UNNAMED '
    '--add-opens=java.base/java.util=ALL-UNNAMED '
    '--add-opens=java.base/java.util.concurrent=ALL-UNNAMED '
    '--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED '
    '--add-opens=java.base/sun.nio.ch=ALL-UNNAMED '
    '--add-opens=java.base/sun.nio.cs=ALL-UNNAMED '
    '--add-opens=java.base/jdk.internal.ref=ALL-UNNAMED'
)
os.environ['JDK_JAVA_OPTIONS'] = jvm_flags

# 3. GCP Project ID 및 Bucket Name 동적 감지
credentials, auth_project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/cloud-platform']
)
if not credentials.valid:
    credentials.refresh(google.auth.transport.requests.Request())

PROJECT_ID = os.getenv("GCP_PROJECT_ID") or auth_project or "your-gcp-project-id"
BUCKET_NAME = os.getenv("GCS_BUCKET_NAME") or f"{PROJECT_ID}-bq-iceberg-demo-bucket"
TOKEN_STR = credentials.token or ""

# 4. GCS Connector Jar 및 ADC 경로 탐색 (상대 경로 사용으로 경로 내 쉼표 문자 분할 이슈 방지)
ADC_PATH = os.path.expanduser("~/.config/gcloud/application_default_credentials.json")
GCS_JAR = "gcs-connector-latest.jar"

with open(ADC_PATH) as f:
    adc_data = json.load(f)

CLIENT_ID = adc_data.get("client_id", "")
CLIENT_SECRET = adc_data.get("client_secret", "")
REFRESH_TOKEN = adc_data.get("refresh_token", "")

print(f"✅ JAVA_HOME     : {os.environ.get('JAVA_HOME')}")
print(f"✅ GCP Project ID : {PROJECT_ID}")
print(f"✅ GCS Bucket Name: gs://{BUCKET_NAME}")
print(f"✅ GCS Jar Path   : {GCS_JAR}")

✅ JAVA_HOME     : /usr/lib/jvm/java-21-openjdk-amd64
✅ GCP Project ID : undertail
✅ GCS Bucket Name: gs://undertail-bq-iceberg-dml-bucket
✅ GCS Jar Path   : gcs-connector-latest.jar


In [4]:
from pyspark.sql import SparkSession

CATALOG_NAME = "lakehouse_catalog"
WAREHOUSE_URI = f"bl://projects/{PROJECT_ID}/catalogs/{BUCKET_NAME}"
REST_URI = "https://biglake.googleapis.com/iceberg/v1/restcatalog"

# 기존 SparkSession이 켜져있다면 중지 (Extensions 적용을 위해)
try:
    SparkSession.getActiveSession().stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("BigLakeSparkSQLGuide")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.codegen.wholeStage", "false")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.jars", GCS_JAR)
    .config("spark.driver.extraClassPath", GCS_JAR)
    .config("spark.executor.extraClassPath", GCS_JAR)
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    .config("spark.hadoop.fs.gs.auth.type", "USER_CREDENTIALS")
    .config("spark.hadoop.fs.gs.auth.client.id", CLIENT_ID)
    .config("spark.hadoop.fs.gs.auth.client.secret", CLIENT_SECRET)
    .config("spark.hadoop.fs.gs.auth.refresh.token", REFRESH_TOKEN)
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.rest.RESTCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", REST_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", WAREHOUSE_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.header.x-goog-user-project", PROJECT_ID)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.header.X-Iceberg-Access-Delegation", "vended-credentials")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.token", TOKEN_STR)
    .getOrCreate()
)

print(f"🚀 SparkSession initialized successfully with Iceberg Extensions!")
print(f"   Catalog Name : {CATALOG_NAME}")
print(f"   REST Catalog : {REST_URI}")

NOTE: Picked up JDK_JAVA_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED
NOTE: Picked up JDK_JAVA_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.conc

:: loading settings :: url = jar:file:/usr/local/google/home/dhmoon/Code/2026-07-22,Netmarble/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /usr/local/google/home/dhmoon/.ivy2/cache
The jars for the packages stored in: /usr/local/google/home/dhmoon/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8157b3b0-6d31-43ea-9b58-3bd42c66b306;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
:: resolution report :: resolve 112ms :: artifacts dl 4ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	------------------------------------------------------------------

🚀 SparkSession initialized successfully with Iceberg Extensions!
   Catalog Name : lakehouse_catalog
   REST Catalog : https://biglake.googleapis.com/iceberg/v1/restcatalog


In [ ]:
print("--- 1. SHOW NAMESPACES ---")
spark.sql(f"SHOW NAMESPACES IN {CATALOG_NAME}").show()

print("\n--- 2. SHOW TABLES IN DEFAULT NAMESPACE ---")
spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.default").show()

--- 1. SHOW NAMESPACES ---


26/07/18 03:26:25 INFO SharedState: Setting hive.metastore.warehouse.dir ('null') to the value of spark.sql.warehouse.dir.
26/07/18 03:26:25 INFO SharedState: Warehouse path is 'file:/usr/local/google/home/dhmoon/Code/2026-07-22,Netmarble/spark-warehouse'.
26/07/18 03:26:26 WARN RESTSessionCatalog: Iceberg REST client is missing the OAuth2 server URI configuration and defaults to https://biglake.googleapis.com/iceberg/v1/restcatalogv1/oauth/tokens. This automatic fallback will be removed in a future Iceberg release.It is recommended to configure the OAuth2 endpoint using the 'oauth2-server-uri' property to be prepared. This warning will disappear if the OAuth2 endpoint is explicitly configured. See https://github.com/apache/iceberg/issues/10537
26/07/18 03:26:27 INFO CatalogUtil: Loading custom FileIO implementation: org.apache.iceberg.hadoop.HadoopFileIO
26/07/18 03:26:28 INFO CodeGenerator: Code generated in 263.591769 ms
26/07/18 03:26:28 INFO CodeGenerator: Code generated in 13.044

+---------+
|namespace|
+---------+
|  default|
| default2|
+---------+


--- 2. SHOW TABLES IN DEFAULT NAMESPACE ---
+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
|  default|external_dml_weblog|      false|
+---------+-------------------+-----------+



26/07/18 03:26:30 INFO CodeGenerator: Code generated in 29.827239 ms
26/07/18 03:26:30 INFO CodeGenerator: Code generated in 8.715997 ms
26/07/18 03:26:30 INFO CodeGenerator: Code generated in 12.323765 ms


26/07/18 03:26:32 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [6]:
print("--- 3. DESCRIBE ICEBERG TABLE SCHEMA ---")
spark.sql(f"DESCRIBE TABLE {CATALOG_NAME}.default.external_weblog").show(truncate=False)

--- 3. DESCRIBE ICEBERG TABLE SCHEMA ---


26/07/18 03:26:36 WARN ErrorHandlers: Unable to parse error response
java.io.UncheckedIOException: org.apache.iceberg.shaded.com.fasterxml.jackson.databind.exc.MismatchedInputException: No content to map due to end-of-input
 at [Source: (String)""; line: 1, column: 0]
	at org.apache.iceberg.util.JsonUtil.parse(JsonUtil.java:101)
	at org.apache.iceberg.rest.responses.ErrorResponseParser.fromJson(ErrorResponseParser.java:71)
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.parseResponse(ErrorHandlers.java:194)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:181)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:323)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:262)
	at org.apache.iceberg.rest.HTTPClient.get(HTTPClient.java:358)
	at org.apache.iceberg.rest.RESTClient.get(RESTClient.java:96)
	at org.apache.iceberg.rest.RESTClient.get(RESTClient.java:79)
	at org.apache.iceberg.rest.RESTSessionCatalog.loadView(RESTSessionCatalog

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `lakehouse_catalog`.`default`.`external_weblog` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 15;
'DescribeRelation false, [col_name#40, data_type#41, comment#42]
+- 'UnresolvedTableOrView [lakehouse_catalog, default, external_weblog], DESCRIBE TABLE, true


In [ ]:
print("--- 4. SPARK SQL GROUP BY AGGREGATION QUERY ---")
query_sql = f"""
    SELECT 
        event_type, 
        COUNT(*) AS total_events, 
        ROUND(SUM(amount), 2) AS total_amount, 
        ROUND(AVG(amount), 2) AS avg_amount 
    FROM {CATALOG_NAME}.default.external_weblog 
    GROUP BY event_type
    ORDER BY total_events DESC
"""
df_agg = spark.sql(query_sql)
df_agg.show()

--- 4. SPARK SQL GROUP BY AGGREGATION QUERY ---


26/07/18 03:20:09 INFO SparkScanBuilder: Skipping aggregate pushdown: group by aggregation push down is not supported
26/07/18 03:20:09 INFO V2ScanRelationPushDown: 
Output: event_type#302, amount#303
         
26/07/18 03:20:09 INFO SnapshotScan: Scanning table lakehouse_catalog.default.external_weblog snapshot 2875036161433468633 created at 2026-07-18T02:37:01.278+00:00 with filter true
26/07/18 03:20:09 INFO BlockManagerInfo: Removed broadcast_5_piece0 on roguelike.c.googlers.com:46525 in memory (size: 31.2 KiB, free: 2.2 GiB)
26/07/18 03:20:09 INFO BlockManagerInfo: Removed broadcast_7_piece0 on roguelike.c.googlers.com:46525 in memory (size: 16.2 KiB, free: 2.2 GiB)
26/07/18 03:20:09 INFO BlockManagerInfo: Removed broadcast_4_piece0 on roguelike.c.googlers.com:46525 in memory (size: 31.2 KiB, free: 2.2 GiB)
26/07/18 03:20:09 INFO BlockManagerInfo: Removed broadcast_6_piece0 on roguelike.c.googlers.com:46525 in memory (size: 12.1 KiB, free: 2.2 GiB)
26/07/18 03:20:10 INFO BaseDistr

+----------+------------+-------------+----------+
|event_type|total_events| total_amount|avg_amount|
+----------+------------+-------------+----------+
|     CLICK|      750810|3.752449755E7|     49.98|
|      VIEW|      749889|3.740391303E7|     49.88|
|  PURCHASE|      749868|3.748648503E7|     49.99|
|    LOGOUT|      749433|3.753617322E7|     50.09|
+----------+------------+-------------+----------+



26/07/18 03:20:12 INFO Executor: Finished task 0.0 in stage 6.0 (TID 7). 5387 bytes result sent to driver
26/07/18 03:20:12 INFO TaskSetManager: Finished task 0.0 in stage 6.0 (TID 7) in 2065 ms on roguelike.c.googlers.com (executor driver) (4/4)
26/07/18 03:20:12 INFO TaskSchedulerImpl: Removed TaskSet 6.0, whose tasks have all completed, from pool 
26/07/18 03:20:12 INFO DAGScheduler: ShuffleMapStage 6 (showString at NativeMethodAccessorImpl.java:0) finished in 2.074 s
26/07/18 03:20:12 INFO DAGScheduler: looking for newly runnable stages
26/07/18 03:20:12 INFO DAGScheduler: running: Set()
26/07/18 03:20:12 INFO DAGScheduler: waiting: Set()
26/07/18 03:20:12 INFO DAGScheduler: failed: Set()
26/07/18 03:20:12 INFO ShufflePartitionsUtil: For shuffle(2), advisory target size: 67108864, actual target size 1048576, minimum partition size: 1048576
26/07/18 03:20:12 INFO SparkContext: Starting job: showString at NativeMethodAccessorImpl.java:0
26/07/18 03:20:12 INFO DAGScheduler: Got job 5 

In [ ]:
print("--- 5. SPARK SQL PARTITION PRUNING & FILTER QUERY ---")
pruning_sql = f"""
    SELECT 
        event_date,
        event_type, 
        COUNT(*) AS total_events, 
        ROUND(SUM(amount), 2) AS total_amount 
    FROM {CATALOG_NAME}.default.external_weblog 
    WHERE event_date BETWEEN '2026-07-03' AND '2026-07-05'
    GROUP BY event_date, event_type
    ORDER BY event_date ASC, total_events DESC
"""
df_pruning = spark.sql(pruning_sql)
df_pruning.show(15)

--- 5. SPARK SQL PARTITION PRUNING & FILTER QUERY ---


26/07/18 03:20:13 INFO SparkScanBuilder: Evaluating completely on Iceberg side: event_date IS NOT NULL
26/07/18 03:20:13 INFO SparkScanBuilder: Evaluating completely on Iceberg side: event_date >= 20637
26/07/18 03:20:13 INFO SparkScanBuilder: Evaluating completely on Iceberg side: event_date <= 20639
26/07/18 03:20:13 INFO V2ScanRelationPushDown: 
Pushing operators to lakehouse_catalog.default.external_weblog
Pushed Filters: event_date IS NOT NULL, event_date >= 20637, event_date <= 20639
Post-Scan Filters: 
         
26/07/18 03:20:13 INFO SparkScanBuilder: Skipping aggregate pushdown: group by aggregation push down is not supported
26/07/18 03:20:13 INFO V2ScanRelationPushDown: 
Output: event_date#356, event_type#358, amount#359
         
26/07/18 03:20:13 INFO SnapshotScan: Scanning table lakehouse_catalog.default.external_weblog snapshot 2875036161433468633 created at 2026-07-18T02:37:01.278+00:00 with filter ((event_date IS NOT NULL AND event_date >= (5-digit-int)) AND event_date

26/07/18 03:20:13 INFO LoggingMetricsReporter: Received metrics report: ScanReport{tableName=lakehouse_catalog.default.external_weblog, snapshotId=2875036161433468633, filter=((not_null(ref(name="event_date")) and ref(name="event_date") >= "(5-digit-int)") and ref(name="event_date") <= "(5-digit-int)"), schemaId=0, projectedFieldIds=[3, 5, 6], projectedFieldNames=[event_date, event_type, amount], scanMetrics=ScanMetricsResult{totalPlanningDuration=TimerResult{timeUnit=NANOSECONDS, totalDuration=PT0.224833933S, count=1}, resultDataFiles=CounterResult{unit=COUNT, value=3}, resultDeleteFiles=CounterResult{unit=COUNT, value=0}, totalDataManifests=CounterResult{unit=COUNT, value=10}, totalDeleteManifests=CounterResult{unit=COUNT, value=0}, scannedDataManifests=CounterResult{unit=COUNT, value=3}, skippedDataManifests=CounterResult{unit=COUNT, value=7}, totalFileSizeInBytes=CounterResult{unit=BYTES, value=13232160}, totalDeleteFileSizeInBytes=CounterResult{unit=BYTES, value=0}, skippedDataFil

+----------+----------+------------+------------+
|event_date|event_type|total_events|total_amount|
+----------+----------+------------+------------+
|2026-07-03|      VIEW|       75549|   3741992.1|
|2026-07-03|     CLICK|       75144|  3730179.24|
|2026-07-03|    LOGOUT|       74859|  3714839.61|
|2026-07-03|  PURCHASE|       74622|  3692706.69|
|2026-07-04|     CLICK|       75465|  3787315.95|
|2026-07-04|    LOGOUT|       75180|  3743293.86|
|2026-07-04|      VIEW|       75030|  3744430.02|
|2026-07-04|  PURCHASE|       74562|  3770182.32|
|2026-07-05|    LOGOUT|       75693|  3819307.77|
|2026-07-05|  PURCHASE|       75528|  3832346.67|
|2026-07-05|     CLICK|       75297|  3759585.87|
|2026-07-05|      VIEW|       75063|  3770826.03|
+----------+----------+------------+------------+



26/07/18 03:20:15 INFO CodecPool: Got brand-new decompressor [.zstd]
26/07/18 03:20:15 INFO Executor: Finished task 0.0 in stage 9.0 (TID 12). 5344 bytes result sent to driver
26/07/18 03:20:15 INFO TaskSetManager: Finished task 0.0 in stage 9.0 (TID 12) in 1682 ms on roguelike.c.googlers.com (executor driver) (1/1)
26/07/18 03:20:15 INFO TaskSchedulerImpl: Removed TaskSet 9.0, whose tasks have all completed, from pool 
26/07/18 03:20:15 INFO DAGScheduler: ShuffleMapStage 9 (showString at NativeMethodAccessorImpl.java:0) finished in 1.690 s
26/07/18 03:20:15 INFO DAGScheduler: looking for newly runnable stages
26/07/18 03:20:15 INFO DAGScheduler: running: Set()
26/07/18 03:20:15 INFO DAGScheduler: waiting: Set()
26/07/18 03:20:15 INFO DAGScheduler: failed: Set()
26/07/18 03:20:15 INFO ShufflePartitionsUtil: For shuffle(3), advisory target size: 67108864, actual target size 1048576, minimum partition size: 1048576
26/07/18 03:20:15 INFO SparkContext: Starting job: showString at NativeMe

## ✍️ [6단계: Spark SQL DML (INSERT, UPDATE, DELETE) 예제 실행]

BigLake REST Catalog로 연동된 Apache Iceberg 테이블(`lakehouse_catalog.default.external_weblog`)에 대해 Spark SQL 엔진을 통해 **INSERT INTO**, **UPDATE**, **DELETE** DML 쿼리를 직접 수행하고 결과를 검증합니다.


In [ ]:
# =========================================================================
# ✍️ [6-1단계] Spark SQL INSERT INTO 쿼리 실행
# =========================================================================
print("--- 6-1. SPARK SQL INSERT INTO ---")
insert_sql = f"""
    INSERT INTO {CATALOG_NAME}.default.external_weblog (
        event_id,
        event_timestamp,
        event_date,
        user_id,
        event_type,
        amount,
        device_os
    )
    VALUES (
        'EVT_SPARK_DML_1001',
        CAST('2026-07-05 15:30:00' AS TIMESTAMP_NTZ),
        DATE '2026-07-05',
        'USER_10500',
        'PURCHASE',
        250.00,
        'IOS'
    )
"""
spark.sql(insert_sql)
print("✅ INSERT INTO 완료. 추가 데이터 확인:")
spark.sql(f"SELECT * FROM {CATALOG_NAME}.default.external_weblog WHERE event_id = 'EVT_SPARK_DML_1001'").show()

# =========================================================================
# ✏️ [6-2단계] Spark SQL UPDATE 쿼리 실행
# =========================================================================
print("\n--- 6-2. SPARK SQL UPDATE ---")
update_sql = f"""
    UPDATE {CATALOG_NAME}.default.external_weblog
    SET amount = 300.00, device_os = 'ANDROID'
    WHERE event_id = 'EVT_SPARK_DML_1001'
"""
spark.sql(update_sql)
print("✅ UPDATE 완료. 수정 데이터 확인:")
spark.sql(f"SELECT * FROM {CATALOG_NAME}.default.external_weblog WHERE event_id = 'EVT_SPARK_DML_1001'").show()

# =========================================================================
# 🗑️ [6-3단계] Spark SQL DELETE 쿼리 실행
# =========================================================================
print("\n--- 6-3. SPARK SQL DELETE ---")
delete_sql = f"""
    DELETE FROM {CATALOG_NAME}.default.external_weblog
    WHERE event_id = 'EVT_SPARK_DML_1001'
"""
spark.sql(delete_sql)
print("✅ DELETE 완료. 삭제 확인 (0건 반환 예상):")
spark.sql(f"SELECT COUNT(*) AS total_count FROM {CATALOG_NAME}.default.external_weblog WHERE event_id = 'EVT_SPARK_DML_1001'").show()


## 🛠️ [7단계: Apache Iceberg 테이블 유지보수 (Compaction & Expire Snapshots)]

DML(INSERT, UPDATE, DELETE) 수행으로 인해 발생한 소형 파키 파일(Small Parquet Files) 및 Delete Delta 파일들을 병합하는 **Compaction (`rewrite_data_files`)** 및 구버전 스냅샷과 고아 파일을 정리하는 **`expire_snapshots`** Spark SQL System Procedure를 실행합니다.


In [ ]:
import datetime

# =========================================================================
# 🛠️ [7-1단계] Apache Iceberg Data Files Compaction (rewrite_data_files)
# =========================================================================
# DML 작업 후 생성된 파편화된 소형 파키 파일(Small Files)을 하나의 파일로 병합하여
# 쿼리 성능(Vectorized Read Performance)을 극대화합니다.
print("--- 7-1. SPARK SQL ICEBERG REWRITE DATA FILES (COMPACTION) ---")
spark.sql(f"""
CALL {CATALOG_NAME}.system.rewrite_data_files(
    table => '{CATALOG_NAME}.default.external_weblog',
    options => map('rewrite-all', 'true')
)
""").show()

# =========================================================================
# 🧹 [7-2단계] Apache Iceberg Legacy Snapshots & Orphan Files Expire (expire_snapshots)
# =========================================================================
# 현재 시각 이전에 생성된 구버전 스냅샷과 불필요한 메타데이터/고아 파일을 정리하고,
# 최신 1개의 활성 스냅샷만 보존(retain_last => 1)하여 GCS 스토리지 용량을 최적화합니다.
print("\n--- 7-2. SPARK SQL ICEBERG EXPIRE SNAPSHOTS ---")
now_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

spark.sql(f"""
CALL {CATALOG_NAME}.system.expire_snapshots(
    table => '{CATALOG_NAME}.default.external_weblog',
    older_than => TIMESTAMP '{now_str}',
    retain_last => 1
)
""").show()


## 💡 Key Takeaways & Best Practices

1. **BigLake REST Catalog URL & Warehouse Format**:
   - REST Catalog URI: `https://biglake.googleapis.com/iceberg/v1/restcatalog`
   - Warehouse URI: `bl://projects/{PROJECT_ID}/catalogs/{BUCKET_NAME}`
2. **GCS Hadoop Connector Authentication**:
   - `spark.hadoop.fs.gs.impl`: `com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem`
   - `spark.hadoop.fs.gs.auth.type`: `USER_CREDENTIALS`
3. **Engine Optimization & Dynamic Security**:
   - JVM 17/21+ 호환성을 위해 `JDK_JAVA_OPTIONS` `--add-opens` 모듈 개방 필수.
   - WholeStageCodegen 조절 (`spark.sql.codegen.wholeStage: false`)을 통해 대규모 분산 쿼리 시 JVM JIT 시그니처 충돌 방지 및 안전한 분산 처리 보장.